## 0. Install Dependencies

In [25]:
%pip install -r requirements.txt

   ---------------------------------------- 0.0/14.6 MB ? eta -:--:--
   --------- ------------------------------ 3.4/14.6 MB 16.8 MB/s eta 0:00:01
   ----------------- ---------------------- 6.6/14.6 MB 16.2 MB/s eta 0:00:01
   ------------------------- -------------- 9.4/14.6 MB 15.4 MB/s eta 0:00:01
   ----------------------------------- ---- 13.1/14.6 MB 15.9 MB/s eta 0:00:01
   ---------------------------------------- 14.6/14.6 MB 14.7 MB/s  0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 15.7 MB/s  0:00:00
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 18.2 MB/s  0:00:00
   ---------------------------------------- 0.0/921.1 kB ? eta -:--:--
   ---------------------------------------- 921.1/921.1 kB 18.4 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ----------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
os.environ["HF_TOKEN"] = "[INSERT TOKEN HERE]"

---
## Part 1 – Local Knowledge Base Retriever (`retriever.py`)

### BM25 from scratch

We implement **Okapi BM25** without any third-party retrieval library.  
The scoring formula per document $D$ and query $Q$ is:

$$\text{score}(D,Q) = \sum_{q_i \in Q} \text{IDF}(q_i) \cdot \frac{f(q_i,D)\,(k_1+1)}{f(q_i,D) + k_1\left(1 - b + b\,\frac{|D|}{\text{avgdl}}\right)}$$

with $k_1 = 1.5$, $b = 0.75$, and smoothed IDF: $\text{IDF}(q_i) = \log\!\left(\frac{N - n(q_i) + 0.5}{n(q_i) + 0.5} + 1\right)$

In [27]:
import importlib, retriever
importlib.reload(retriever)

from retriever import DOCUMENTS, retrieve, retrieve_with_scores, BM25_RETRIEVER

print(f"Total documents      : {len(DOCUMENTS)}")
print(f"Avg doc length (tok) : {BM25_RETRIEVER.avgdl:.1f}")

Loading invitees dataset …
  → 3 documents loaded.
BM25 index built.

Total documents      : 3
Avg doc length (tok) : 47.0


### Manual Retrieval Verification (5 queries)

In [28]:
def show_results(query, k=3):
    results = retrieve_with_scores(query, k=k)
    print(f"\n{'='*62}")
    print(f"QUERY : {query}")
    print(f"{'='*62}")
    for rank, (doc, score) in enumerate(results, 1):
        print(f"\n  -- Result {rank}  (BM25 score: {score:.4f}) --")
        for line in doc.page_content.split('\n'):
            print(f"     {line}")
    print(f"\n  >>> Best match: {results[0][0].metadata['name']}")

# Query 1 – direct name lookup
show_results("Ada Lovelace")


QUERY : Ada Lovelace

  -- Result 1  (BM25 score: 3.1521) --
     Name: Ada Lovelace
     Relation: best friend
     Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.
     Email: ada.lovelace@example.com

  -- Result 2  (BM25 score: 0.0000) --
     Name: Dr. Nikola Tesla
     Relation: old friend from university days
     Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk.
     Email: nikola.tesla@gmail.com

  -- Result 3  (BM25 score: 0.0000) --
     Name: Marie Curie
     Relation: no relation
     Description: Marie Curie was a groundbreaking physici

In [29]:
# Query 2 – relation keyword
show_results("old friend university days")


QUERY : old friend university days

  -- Result 1  (BM25 score: 4.4490) --
     Name: Dr. Nikola Tesla
     Relation: old friend from university days
     Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk.
     Email: nikola.tesla@gmail.com

  -- Result 2  (BM25 score: 0.7552) --
     Name: Ada Lovelace
     Relation: best friend
     Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.
     Email: ada.lovelace@example.com

  -- Result 3  (BM25 score: 0.0000) --
     Name: Marie Curie
     Relation: no relation
     Description: Marie Curie was a groundbr

In [30]:
# Query 3 – scientific field keywords
show_results("physicist radioactivity Nobel Prize scientist")


QUERY : physicist radioactivity Nobel Prize scientist

  -- Result 1  (BM25 score: 2.4553) --
     Name: Marie Curie
     Relation: no relation
     Description: Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity.
     Email: marie.curie@example.com

  -- Result 2  (BM25 score: 0.0000) --
     Name: Ada Lovelace
     Relation: best friend
     Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.
     Email: ada.lovelace@example.com

  -- Result 3  (BM25 score: 0.0000) --
     Name: Dr. Nikola Tesla
     Relation: old friend from university days
     Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it wit

In [31]:
# Query 4 – contact info
show_results("gmail email contact")


QUERY : gmail email contact

  -- Result 1  (BM25 score: 0.9826) --
     Name: Dr. Nikola Tesla
     Relation: old friend from university days
     Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk.
     Email: nikola.tesla@gmail.com

  -- Result 2  (BM25 score: 0.1671) --
     Name: Marie Curie
     Relation: no relation
     Description: Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity.
     Email: marie.curie@example.com

  -- Result 3  (BM25 score: 0.1251) --
     Name: Ada Lovelace
     Relation: best friend
     Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer pro

In [32]:
# Query 5 – computing / programming keywords
show_results("mathematician programmer computing algorithm")


QUERY : mathematician programmer computing algorithm

  -- Result 1  (BM25 score: 2.7577) --
     Name: Ada Lovelace
     Relation: best friend
     Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.
     Email: ada.lovelace@example.com

  -- Result 2  (BM25 score: 0.0000) --
     Name: Dr. Nikola Tesla
     Relation: old friend from university days
     Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk.
     Email: nikola.tesla@gmail.com

  -- Result 3  (BM25 score: 0.0000) --
     Name: Marie Curie
     Relation: no relation
     Description: Marie Cu

---
## Part 2 – RAG Agent (`tools.py` + `app.py`)

### Tool definition

In [33]:
import importlib, tools
importlib.reload(tools)
from tools import guest_info_tool

print(f"Tool name    : {guest_info_tool.name}")
print(f"Output type  : {guest_info_tool.output_type}")
print(f"Inputs       : {guest_info_tool.inputs}")
print(f"\nDescription:\n{guest_info_tool.description}")

Tool name    : guest_info_retriever
Output type  : string
Inputs       : {'query': {'type': 'string', 'description': "A search query containing the guest's name, relation to the host, or descriptive keywords about the guest."}}

Description:
Retrieves detailed information about gala guests from the local knowledge base. Input should be the guest's name, their relation to the host, or keywords describing them (e.g. 'physicist', 'best friend', 'university days'). Returns the top-3 matching guest records including name, relation, description, and email. ALWAYS call this tool before answering any question about a specific guest.


In [34]:
# Quick tool sanity-check
print(guest_info_tool.forward("Ada Lovelace"))

[Result 1]
Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com

[Result 2]
Name: Dr. Nikola Tesla
Relation: old friend from university days
Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about pigeons, so that might make for good small talk.
Email: nikola.tesla@gmail.com

[Result 3]
Name: Marie Curie
Relation: no relation
Description: Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity.
Email: marie.curie@example.com


### Agent instantiation

In [40]:
from smolagents import CodeAgent, LiteLLMModel
from app import SYSTEM_PROMPT

model = LiteLLMModel(
    model_id="huggingface/Qwen/Qwen2.5-72B-Instruct",
)

alfred = CodeAgent(
    tools=[guest_info_tool],
    model=model,
    max_steps=3,
)

SYSTEM_PROMPT = """\
You are Alfred, an intelligent butler assistant for the most extravagant gala
of the century.  Your job is to help the host by answering questions about
the guests attending the event.

Rules you must follow:
1. Whenever a question is about a guest (their background, relation, email,
   or any personal detail), you MUST call the guest_info_retriever tool first.
2. Base your final answer ONLY on information returned by the tool.
3. Structure every final answer exactly as shown below:

   Answer: <your answer in 1-3 sentences>
   Evidence:
     - "<short quote or key fact from retrieved Result 1>"
     - "<short quote or key fact from retrieved Result 2>"
     (add a third bullet if useful)

4. If the tool returns no relevant match, respond:
   "This information is not found in the knowledge base."
5. For questions that are clearly out of scope (no guest involved), answer
   from your own knowledge without calling the retrieval tool.
"""

print("Alfred (RAG Agent) ready.")

Alfred (RAG Agent) ready.


### Demo prompts (5 total: 3 answerable, 2 out-of-scope)

In [41]:
# Demo 1 — Answerable
p = "Tell me about our guest named Lady Ada Lovelace."
print(f"{SYSTEM_PROMPT} \nPROMPT: {p}\n")
print(alfred.run(f"{SYSTEM_PROMPT} \nPROMPT: {p}\n"))

You are Alfred, an intelligent butler assistant for the most extravagant gala
of the century.  Your job is to help the host by answering questions about
the guests attending the event.

Rules you must follow:
1. Whenever a question is about a guest (their background, relation, email,
   or any personal detail), you MUST call the guest_info_retriever tool first.
2. Base your final answer ONLY on information returned by the tool.
3. Structure every final answer exactly as shown below:

   Answer: <your answer in 1-3 sentences>
   Evidence:
     - "<short quote or key fact from retrieved Result 1>"
     - "<short quote or key fact from retrieved Result 2>"
     (add a third bullet if useful)

4. If the tool returns no relevant match, respond:
   "This information is not found in the knowledge base."
5. For questions that are clearly out of scope (no guest involved), answer
   from your own knowledge without calling the retrieval tool.
 
PROMPT: Tell me about our guest named Lady Ada Lovel

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are Alfred, an intelligent butler assistant for the most extravagant gala                                   │
│ of the century.  Your job is to help the host by answering questions about                                      │
│ the guests attending the event.                                                                                 │
│                                                                                                                 │
│ Rules you must follow:                                                                                          │
│ 1. Whenever a question is about a guest (their background, relation, email,                                     │
│    or any personal detail), you MUST call the guest_info_retriever tool first.                                  │
│ 2. Base your final answer ONLY on information returned by the tool.                                             │
│ 3. Structure every final answer exactly as shown below:                                                         │
│                                                                                                                 │
│    Answer: <your answer in 1-3 sentences>                                                                       │
│    Evidence:                                                                                                    │
│      - "<short quote or key fact from retrieved Result 1>"                                                      │
│      - "<short quote or key fact from retrieved Result 2>"                                                      │
│      (add a third bullet if useful)                                                                             │
│                                                                                                                 │
│ 4. If the tool returns no relevant match, respond:                                                              │
│    "This information is not found in the knowledge base."                                                       │
│ 5. For questions that are clearly out of scope (no guest involved), answer                                      │
│    from your own knowledge without calling the retrieval tool.                                                  │
│                                                                                                                 │
│ PROMPT: Tell me about our guest named Lady Ada Lovelace.                                                        │
│                                                                                                                 │
╰─ LiteLLMModel - huggingface/Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  guest_info = guest_info_retriever(query="Lady Ada Lovelace")                                                     
  print(guest_info)                                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[Result 1]
Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for 
her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work
on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com

[Result 2]
Name: Dr. Nikola Tesla
Relation: old friend from university days
Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless 
energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about 
pigeons, so that might make for good small talk.
Email: nikola.tesla@gmail.com

[Result 3]
Name: Marie Curie
Relation: no relation
Description: Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity.
Email: marie.curie@example.com

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(f"""Answer: Lady Ada Lovelace is the host's best friend and an esteemed mathematician, known for    
  her pioneering work in mathematics and computing.                                                                
  Evidence:                                                                                                        
    - "Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend."                          
    - "She is renowned for her pioneering work in mathematics and computing, often celebrated as the first         
  computer programmer due to her work on Charles Babbage's Analytical Engine." """)                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Answer: Lady Ada Lovelace is the host's best friend and an esteemed mathematician, known for her 
pioneering work in mathematics and computing.
Evidence:
  - "Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend."
  - "She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer 
programmer due to her work on Charles Babbage's Analytical Engine." 

[Step 2: Duration 4.14 seconds| Input tokens: 5,087 | Output tokens: 168]

Answer: Lady Ada Lovelace is the host's best friend and an esteemed mathematician, known for her pioneering work in mathematics and computing.
Evidence:
  - "Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend."
  - "She is renowned for her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work on Charles Babbage's Analytical Engine." 


In [48]:
# Demo 2 — Answerable
p = "What is Dr. Nikola Tesla's email address?"
print(f"{SYSTEM_PROMPT} \nPROMPT: {p}\n")
print(alfred.run(f"{SYSTEM_PROMPT} \nPROMPT: {p}\n"))

You are Alfred, an intelligent butler assistant for the most extravagant gala
of the century.  Your job is to help the host by answering questions about
the guests attending the event.

Rules you must follow:
1. Whenever a question is about a guest (their background, relation, email,
   or any personal detail), you MUST call the guest_info_retriever tool first.
2. Base your final answer ONLY on information returned by the tool.
3. Structure every final answer exactly as shown below:

   Answer: <your answer in 1-3 sentences>
   Evidence:
     - "<short quote or key fact from retrieved Result 1>"
     - "<short quote or key fact from retrieved Result 2>"
     (add a third bullet if useful)

4. If the tool returns no relevant match, respond:
   "This information is not found in the knowledge base."
5. For questions that are clearly out of scope (no guest involved), answer
   from your own knowledge without calling the retrieval tool.
 
PROMPT: What is Dr. Nikola Tesla's email address?



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are Alfred, an intelligent butler assistant for the most extravagant gala                                   │
│ of the century.  Your job is to help the host by answering questions about                                      │
│ the guests attending the event.                                                                                 │
│                                                                                                                 │
│ Rules you must follow:                                                                                          │
│ 1. Whenever a question is about a guest (their background, relation, email,                                     │
│    or any personal detail), you MUST call the guest_info_retriever tool first.                                  │
│ 2. Base your final answer ONLY on information returned by the tool.                                             │
│ 3. Structure every final answer exactly as shown below:                                                         │
│                                                                                                                 │
│    Answer: <your answer in 1-3 sentences>                                                                       │
│    Evidence:                                                                                                    │
│      - "<short quote or key fact from retrieved Result 1>"                                                      │
│      - "<short quote or key fact from retrieved Result 2>"                                                      │
│      (add a third bullet if useful)                                                                             │
│                                                                                                                 │
│ 4. If the tool returns no relevant match, respond:                                                              │
│    "This information is not found in the knowledge base."                                                       │
│ 5. For questions that are clearly out of scope (no guest involved), answer                                      │
│    from your own knowledge without calling the retrieval tool.                                                  │
│                                                                                                                 │
│ PROMPT: What is Dr. Nikola Tesla's email address?                                                               │
│                                                                                                                 │
╰─ LiteLLMModel - huggingface/Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  guest_info = guest_info_retriever(query="Dr. Nikola Tesla")                                                      
  print(guest_info)                                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[Result 1]
Name: Dr. Nikola Tesla
Relation: old friend from university days
Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless 
energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about 
pigeons, so that might make for good small talk.
Email: nikola.tesla@gmail.com

[Result 2]
Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for 
her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work
on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com

[Result 3]
Name: Marie Curie
Relation: no relation
Description: Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity.
Email: marie.curie@example.com

Out: None

...

Out: None

[Step 3: Duration 3.69 seconds| Input tokens: 8,068 | Output tokens: 214]

Reached max steps.

[Step 4: Duration 2.25 seconds| Input tokens: 9,298 | Output tokens: 257]

Answer: Dr. Nikola Tesla's email address is nikola.tesla@gmail.com.
Evidence:
  - "Name: Dr. Nikola Tesla"
  - "Email: nikola.tesla@gmail.com"


In [49]:
# Demo 3 — Answerable
p = "How is Marie Curie related to the host, and what is she famous for?"
print(f"{SYSTEM_PROMPT} \nPROMPT: {p}\n")
print(alfred.run(f"{SYSTEM_PROMPT} \nPROMPT: {p}\n"))

You are Alfred, an intelligent butler assistant for the most extravagant gala
of the century.  Your job is to help the host by answering questions about
the guests attending the event.

Rules you must follow:
1. Whenever a question is about a guest (their background, relation, email,
   or any personal detail), you MUST call the guest_info_retriever tool first.
2. Base your final answer ONLY on information returned by the tool.
3. Structure every final answer exactly as shown below:

   Answer: <your answer in 1-3 sentences>
   Evidence:
     - "<short quote or key fact from retrieved Result 1>"
     - "<short quote or key fact from retrieved Result 2>"
     (add a third bullet if useful)

4. If the tool returns no relevant match, respond:
   "This information is not found in the knowledge base."
5. For questions that are clearly out of scope (no guest involved), answer
   from your own knowledge without calling the retrieval tool.
 
PROMPT: How is Marie Curie related to the host, and 

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are Alfred, an intelligent butler assistant for the most extravagant gala                                   │
│ of the century.  Your job is to help the host by answering questions about                                      │
│ the guests attending the event.                                                                                 │
│                                                                                                                 │
│ Rules you must follow:                                                                                          │
│ 1. Whenever a question is about a guest (their background, relation, email,                                     │
│    or any personal detail), you MUST call the guest_info_retriever tool first.                                  │
│ 2. Base your final answer ONLY on information returned by the tool.                                             │
│ 3. Structure every final answer exactly as shown below:                                                         │
│                                                                                                                 │
│    Answer: <your answer in 1-3 sentences>                                                                       │
│    Evidence:                                                                                                    │
│      - "<short quote or key fact from retrieved Result 1>"                                                      │
│      - "<short quote or key fact from retrieved Result 2>"                                                      │
│      (add a third bullet if useful)                                                                             │
│                                                                                                                 │
│ 4. If the tool returns no relevant match, respond:                                                              │
│    "This information is not found in the knowledge base."                                                       │
│ 5. For questions that are clearly out of scope (no guest involved), answer                                      │
│    from your own knowledge without calling the retrieval tool.                                                  │
│                                                                                                                 │
│ PROMPT: How is Marie Curie related to the host, and what is she famous for?                                     │
│                                                                                                                 │
╰─ LiteLLMModel - huggingface/Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  info = guest_info_retriever(query="Marie Curie")                                                                 
  print(info)                                                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[Result 1]
Name: Marie Curie
Relation: no relation
Description: Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity.
Email: marie.curie@example.com

[Result 2]
Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for 
her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work
on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com

[Result 3]
Name: Dr. Nikola Tesla
Relation: old friend from university days
Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless 
energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about 
pigeons, so that might make for good small talk.
Email: nikola.tesla@gmail.com

Out: None

...

Code execution failed at line 'description = info[0\].get('Description')' due to: InterpreterError: Object [ has no
attribute get

[Step 3: Duration 4.41 seconds| Input tokens: 8,143 | Output tokens: 262]

Reached max steps.

[Step 4: Duration 2.86 seconds| Input tokens: 9,661 | Output tokens: 320]

Answer: Marie Curie has no direct relation to the host, and she is famous for her groundbreaking research on radioactivity.
Evidence:
  - "Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity."
  - "Relation: no relation"


In [50]:
# Demo 4 — Out-of-scope (guest not in KB)
p = "Can you tell me about the guest Albert Einstein?"
print(f"{SYSTEM_PROMPT} \nPROMPT: {p}\n")
print(alfred.run(f"{SYSTEM_PROMPT} \nPROMPT: {p}\n"))

You are Alfred, an intelligent butler assistant for the most extravagant gala
of the century.  Your job is to help the host by answering questions about
the guests attending the event.

Rules you must follow:
1. Whenever a question is about a guest (their background, relation, email,
   or any personal detail), you MUST call the guest_info_retriever tool first.
2. Base your final answer ONLY on information returned by the tool.
3. Structure every final answer exactly as shown below:

   Answer: <your answer in 1-3 sentences>
   Evidence:
     - "<short quote or key fact from retrieved Result 1>"
     - "<short quote or key fact from retrieved Result 2>"
     (add a third bullet if useful)

4. If the tool returns no relevant match, respond:
   "This information is not found in the knowledge base."
5. For questions that are clearly out of scope (no guest involved), answer
   from your own knowledge without calling the retrieval tool.
 
PROMPT: Can you tell me about the guest Albert Einst

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are Alfred, an intelligent butler assistant for the most extravagant gala                                   │
│ of the century.  Your job is to help the host by answering questions about                                      │
│ the guests attending the event.                                                                                 │
│                                                                                                                 │
│ Rules you must follow:                                                                                          │
│ 1. Whenever a question is about a guest (their background, relation, email,                                     │
│    or any personal detail), you MUST call the guest_info_retriever tool first.                                  │
│ 2. Base your final answer ONLY on information returned by the tool.                                             │
│ 3. Structure every final answer exactly as shown below:                                                         │
│                                                                                                                 │
│    Answer: <your answer in 1-3 sentences>                                                                       │
│    Evidence:                                                                                                    │
│      - "<short quote or key fact from retrieved Result 1>"                                                      │
│      - "<short quote or key fact from retrieved Result 2>"                                                      │
│      (add a third bullet if useful)                                                                             │
│                                                                                                                 │
│ 4. If the tool returns no relevant match, respond:                                                              │
│    "This information is not found in the knowledge base."                                                       │
│ 5. For questions that are clearly out of scope (no guest involved), answer                                      │
│    from your own knowledge without calling the retrieval tool.                                                  │
│                                                                                                                 │
│ PROMPT: Can you tell me about the guest Albert Einstein?                                                        │
│                                                                                                                 │
╰─ LiteLLMModel - huggingface/Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  guest_info = guest_info_retriever(query="Albert Einstein")                                                       
  print(guest_info)                                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[Result 1]
Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for 
her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work
on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com

[Result 2]
Name: Dr. Nikola Tesla
Relation: old friend from university days
Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless 
energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about 
pigeons, so that might make for good small talk.
Email: nikola.tesla@gmail.com

[Result 3]
Name: Marie Curie
Relation: no relation
Description: Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity.
Email: marie.curie@example.com

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("This information is not found in the knowledge base.")                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: This information is not found in the knowledge base.

[Step 2: Duration 2.25 seconds| Input tokens: 5,076 | Output tokens: 87]

This information is not found in the knowledge base.


In [51]:
# Demo 5 — Out-of-scope (general knowledge)
p = "What is the capital of France?"
print(f"{SYSTEM_PROMPT} \nPROMPT: {p}\n")
print(alfred.run(f"{SYSTEM_PROMPT} \nPROMPT: {p}\n"))

You are Alfred, an intelligent butler assistant for the most extravagant gala
of the century.  Your job is to help the host by answering questions about
the guests attending the event.

Rules you must follow:
1. Whenever a question is about a guest (their background, relation, email,
   or any personal detail), you MUST call the guest_info_retriever tool first.
2. Base your final answer ONLY on information returned by the tool.
3. Structure every final answer exactly as shown below:

   Answer: <your answer in 1-3 sentences>
   Evidence:
     - "<short quote or key fact from retrieved Result 1>"
     - "<short quote or key fact from retrieved Result 2>"
     (add a third bullet if useful)

4. If the tool returns no relevant match, respond:
   "This information is not found in the knowledge base."
5. For questions that are clearly out of scope (no guest involved), answer
   from your own knowledge without calling the retrieval tool.
 
PROMPT: What is the capital of France?



╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are Alfred, an intelligent butler assistant for the most extravagant gala                                   │
│ of the century.  Your job is to help the host by answering questions about                                      │
│ the guests attending the event.                                                                                 │
│                                                                                                                 │
│ Rules you must follow:                                                                                          │
│ 1. Whenever a question is about a guest (their background, relation, email,                                     │
│    or any personal detail), you MUST call the guest_info_retriever tool first.                                  │
│ 2. Base your final answer ONLY on information returned by the tool.                                             │
│ 3. Structure every final answer exactly as shown below:                                                         │
│                                                                                                                 │
│    Answer: <your answer in 1-3 sentences>                                                                       │
│    Evidence:                                                                                                    │
│      - "<short quote or key fact from retrieved Result 1>"                                                      │
│      - "<short quote or key fact from retrieved Result 2>"                                                      │
│      (add a third bullet if useful)                                                                             │
│                                                                                                                 │
│ 4. If the tool returns no relevant match, respond:                                                              │
│    "This information is not found in the knowledge base."                                                       │
│ 5. For questions that are clearly out of scope (no guest involved), answer                                      │
│    from your own knowledge without calling the retrieval tool.                                                  │
│                                                                                                                 │
│ PROMPT: What is the capital of France?                                                                          │
│                                                                                                                 │
╰─ LiteLLMModel - huggingface/Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Paris")                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Paris

[Step 1: Duration 1.99 seconds| Input tokens: 2,369 | Output tokens: 37]

Paris


## Part 3 – Mini Evaluation

In [53]:
from app import EVAL_SET

for item in EVAL_SET:
    label = "ANSWERABLE  " if item["answerable"] else "UNANSWERABLE"
    print(f"\nQ{item['id']:02d} [{label}] {item['question']}")
    print("-" * 68)
    print(alfred.run(f"{SYSTEM_PROMPT} \nPROMPT: {item["question"]}\n"))


Q01 [ANSWERABLE  ] Who is Ada Lovelace and how does she know the host?
--------------------------------------------------------------------


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are Alfred, an intelligent butler assistant for the most extravagant gala                                   │
│ of the century.  Your job is to help the host by answering questions about                                      │
│ the guests attending the event.                                                                                 │
│                                                                                                                 │
│ Rules you must follow:                                                                                          │
│ 1. Whenever a question is about a guest (their background, relation, email,                                     │
│    or any personal detail), you MUST call the guest_info_retriever tool first.                                  │
│ 2. Base your final answer ONLY on information returned by the tool.                                             │
│ 3. Structure every final answer exactly as shown below:                                                         │
│                                                                                                                 │
│    Answer: <your answer in 1-3 sentences>                                                                       │
│    Evidence:                                                                                                    │
│      - "<short quote or key fact from retrieved Result 1>"                                                      │
│      - "<short quote or key fact from retrieved Result 2>"                                                      │
│      (add a third bullet if useful)                                                                             │
│                                                                                                                 │
│ 4. If the tool returns no relevant match, respond:                                                              │
│    "This information is not found in the knowledge base."                                                       │
│ 5. For questions that are clearly out of scope (no guest involved), answer                                      │
│    from your own knowledge without calling the retrieval tool.                                                  │
│                                                                                                                 │
│ PROMPT: Who is Ada Lovelace and how does she know the host?                                                     │
│                                                                                                                 │
╰─ LiteLLMModel - huggingface/Qwen/Qwen2.5-72B-Instruct ──────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  guest_info = guest_info_retriever(query="Ada Lovelace")                                                          
  print(guest_info)                                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[Result 1]
Name: Ada Lovelace
Relation: best friend
Description: Lady Ada Lovelace is my best friend. She is an esteemed mathematician and friend. She is renowned for 
her pioneering work in mathematics and computing, often celebrated as the first computer programmer due to her work
on Charles Babbage's Analytical Engine.
Email: ada.lovelace@example.com

[Result 2]
Name: Dr. Nikola Tesla
Relation: old friend from university days
Description: Dr. Nikola Tesla is an old friend from your university days. He's recently patented a new wireless 
energy transmission system and would be delighted to discuss it with you. Just remember he's passionate about 
pigeons, so that might make for good small talk.
Email: nikola.tesla@gmail.com

[Result 3]
Name: Marie Curie
Relation: no relation
Description: Marie Curie was a groundbreaking physicist and chemist, famous for her research on radioactivity.
Email: marie.curie@example.com

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The gala starts at 7:00 PM tonight.")                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The gala starts at 7:00 PM tonight.

[Step 1: Duration 2.29 seconds| Input tokens: 2,370 | Output tokens: 48]

The gala starts at 7:00 PM tonight.


Accuracy among answerable samples: 100%
Accuracy among unanswerable samples: 66%
Total Accuracy: 90%